# Review LLM Judgments

Browse and filter the LLM-judged survey-vote matches.
Each pair has been scored by Mistral (via Ollama) with a relevance score, explanation, and go/no-go verdict.


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

DATA_DIR = Path("../../data")
JUDGED_CSV = DATA_DIR / "matches" / "survey_vote_matches_judged.csv"

df = pd.read_csv(JUDGED_CSV)
print(f"Total judged pairs: {len(df)}")
print(f"Columns: {list(df.columns)}")
df.head()


## Summary statistics


In [ ]:
# Filter out failed parses
valid = df[df["llm_score"] > 0].copy()
print(f"Valid judgments: {len(valid)} / {len(df)}")
print(f"Parse errors: {(df['llm_score'] == -1).sum()}")
print()

# Go / No-Go breakdown
n_go = valid["llm_go"].sum()
n_nogo = len(valid) - n_go
print(f"Go:    {n_go}  ({100 * n_go / len(valid):.1f}%)")
print(f"No-Go: {n_nogo}  ({100 * n_nogo / len(valid):.1f}%)")
print()

# Score stats
print(f"Score distribution:")
print(valid["llm_score"].describe().round(2))


## Score distribution


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram of scores
axes[0].hist(
    valid["llm_score"],
    bins=range(1, 12),
    edgecolor="white",
    color="#4c72b0",
    align="left",
)
axes[0].set_xlabel("LLM Score")
axes[0].set_ylabel("Count")
axes[0].set_title("Distribution of LLM Relevance Scores")
axes[0].set_xticks(range(1, 11))

# Go / No-Go pie chart
go_counts = valid["llm_go"].value_counts()
labels = ["Go ✅" if k else "No-Go ❌" for k in go_counts.index]
colors = ["#55a868" if k else "#c44e52" for k in go_counts.index]
axes[1].pie(
    go_counts.values, labels=labels, colors=colors, autopct="%1.1f%%", startangle=90
)
axes[1].set_title("Go / No-Go Breakdown")

plt.tight_layout()
plt.show()


## Top "Go" matches (highest scored)


In [ ]:
go_df = valid[valid["llm_go"] == True].sort_values("llm_score", ascending=False)

display_cols = [
    "question_text",
    "vote_summary",
    "similarity_score",
    "llm_score",
    "llm_explanation",
    "llm_go",
]

print(f"✅ {len(go_df)} 'Go' matches")
go_df[display_cols].head(20).style.set_properties(
    subset=["question_text", "vote_summary"],
    **{"max-width": "400px", "white-space": "normal"},
)


## Worst-scored matches (sanity check)

These should be clearly unrelated pairs. If they look related, the LLM might need prompt tuning.


In [ ]:
worst = valid.sort_values("llm_score", ascending=True)

print(f"❌ Lowest scored matches:")
worst[display_cols].head(20).style.set_properties(
    subset=["question_text", "vote_summary"],
    **{"max-width": "400px", "white-space": "normal"},
)


## Similarity score vs LLM score

Compare the embedding cosine similarity with the LLM's qualitative judgment.


In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
scatter = ax.scatter(
    valid["similarity_score"],
    valid["llm_score"],
    c=valid["llm_go"].map({True: "#55a868", False: "#c44e52"}),
    alpha=0.4,
    s=15,
)
ax.set_xlabel("Cosine Similarity (embedding)")
ax.set_ylabel("LLM Score (1-10)")
ax.set_title("Embedding Similarity vs. LLM Relevance Score")
ax.axhline(y=7, color="gray", linestyle="--", alpha=0.5, label="Go threshold (7)")
ax.legend(["Go threshold"], loc="lower right")
plt.tight_layout()
plt.show()


## Browse individual pairs

Use this section to inspect specific pairs interactively.


In [ ]:
def show_pair(idx: int):
    """Display a single match pair with its judgment."""
    row = df.iloc[idx]
    print(f"{'=' * 80}")
    print(f"Pair #{idx}")
    print(f"{'=' * 80}")
    print(f"\n📋 SURVEY QUESTION ({row.get('question_id', 'N/A')}):")
    print(f"   {row['question_text'][:500]}")
    print(f"\n🗳️  VOTE ({row.get('vote_id', 'N/A')}):")
    print(f"   {row['vote_summary'][:500]}")
    print(f"\n📊 SCORES:")
    print(f"   Cosine similarity: {row['similarity_score']:.4f}")
    print(f"   LLM score:         {row['llm_score']}/10")
    print(f"   LLM explanation:   {row['llm_explanation']}")
    print(f"   Verdict:           {'✅ GO' if row['llm_go'] else '❌ NO-GO'}")
    print()


# Show first 5 Go matches
for i in go_df.index[:5]:
    show_pair(i)
